# sanskrit

> Devanagari primitives: Vedic marks, daṇḍas, an ASCII fold for any script, citation sigla.

In [ ]:
#| default_exp sanskrit

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

The small text constants and folds the rest of ganapati reads Devanagari with. They began in litesearch, which keeps its own copy for its FTS5 tokenizer; ganapati has them here so that it needs nothing from litesearch to scan a verse.

In [ ]:
#| export
import re, unicodedata
from functools import lru_cache
from fastcore.all import L, Path, ifnone

VEDIC_MARKS = re.compile('['
    '॑-॔'      # udatta, anudatta, devanagari grave/acute
    '᳐-᳹'      # vedic extensions: tone marks, nasalisations
    '꣠-꣰'      # devanagari extended combining digits, stopping before the candrabindu block
    ']')

DANDA, DDANDA = '।', '॥'          # । ॥

_INDEP = {'अ':'a','आ':'a','इ':'i','ई':'i','उ':'u','ऊ':'u','ऋ':'r','ॠ':'r','ऌ':'l','ॡ':'l',
          'ए':'e','ऐ':'ai','ओ':'o','औ':'au','ऍ':'e','ऎ':'e','ऑ':'o','ऒ':'o'}
_CONS  = {'क':'k','ख':'kh','ग':'g','घ':'gh','ङ':'n','च':'c','छ':'ch','ज':'j','झ':'jh','ञ':'n',
          'ट':'t','ठ':'th','ड':'d','ढ':'dh','ण':'n','त':'t','थ':'th','द':'d','ध':'dh','न':'n',
          'प':'p','फ':'ph','ब':'b','भ':'bh','म':'m','य':'y','र':'r','ल':'l','व':'v',
          'श':'s','ष':'s','स':'s','ह':'h','ळ':'l','ऴ':'l','ऱ':'r','ऩ':'n',
          'क़':'k','ख़':'kh','ग़':'g','ज़':'z','ड़':'d','ढ़':'dh','फ़':'f','य़':'y'}
_MATRA = {'ा':'a','ि':'i','ी':'i','ु':'u','ू':'u','ृ':'r','ॄ':'r','ॢ':'l','ॣ':'l',
          'े':'e','ै':'ai','ो':'o','ौ':'au','ॅ':'e','ॆ':'e','ॉ':'o','ॊ':'o'}
# `ꣳ`/`ꣲ`/`꣰` are the Vedic candrabindu signs — a nasal, not an accent, so they fold to `m` and
# line up with the `ṃ` an IAST edition of the same line prints.
_SIGN  = {'ं':'m','ः':'h','ँ':'m','ऽ':'', '़':'', 'ꣲ':'m','ꣳ':'m','ꣴ':'m','ꣵ':'m','ꣶ':'m','ꣷ':'m'}
VIRAMA = '्'
DEVA_DIGITS = {chr(0x966+i): str(i) for i in range(10)}

def strip_vedic(s:str) -> str:
    'Drop Vedic tone marks from Devanagari, leaving Latin diacritics alone. NFC out.'
    if not s: return ''
    return unicodedata.normalize('NFC', VEDIC_MARKS.sub('', unicodedata.normalize('NFD', s)))

def deva2ascii(s:str) -> str:
    'Devanagari to a bare ASCII key, applying the implicit `a` of an unmarked consonant.'
    out, i, n = [], 0, len(s)
    while i < n:
        ch = s[i]
        if ch in _CONS:
            out.append(_CONS[ch]); i += 1
            # look past a nukta that was not pre-composed
            while i < n and s[i] == '़': i += 1
            if i < n and s[i] == VIRAMA: i += 1; continue          # bare consonant, no vowel
            if i < n and s[i] in _MATRA: out.append(_MATRA[s[i]]); i += 1; continue
            out.append('a'); continue                                # implicit vowel
        if ch in _INDEP: out.append(_INDEP[ch]); i += 1; continue
        if ch in _SIGN:  out.append(_SIGN[ch]);  i += 1; continue
        if ch in _MATRA: out.append(_MATRA[ch]); i += 1; continue    # orphan matra
        if ch in DEVA_DIGITS: out.append(DEVA_DIGITS[ch]); i += 1; continue
        if ch == VIRAMA: i += 1; continue
        out.append(ch); i += 1
    return ''.join(out)

def _latn_fold(s:str) -> str:
    'IAST/ISO to ASCII — the same folding `simplify casefold` already applies to a Latin token.'
    d = unicodedata.normalize('NFD', s)
    return ''.join(c for c in d if unicodedata.category(c) != 'Mn').lower()

DEVANAGARI = re.compile('[ऀ-ॿ᳐-᳹꣠-ꣿ]')

def detect_script(s:str) -> str:
    "`'deva'`, `'latn'` or `'other'` — enough to pick a folding path."
    if not s: return 'other'
    d = len(DEVANAGARI.findall(s))
    l = sum(1 for c in s if 'a' <= c.lower() <= 'z' or unicodedata.category(c) == 'Ll')
    if d and d >= l: return 'deva'
    return 'latn' if l else 'other'

@lru_cache(maxsize=1<<16)
def fold_token(s:str) -> str:
    '''The ASCII index key for one token, whatever script it arrived in.

    Cached because it is called once per token per FTS write *and* per query, and a corpus is a
    small vocabulary repeated: 70,413 tokens of the eval corpus are 4,153 distinct strings, so
    94% of the calls are asking a question that has already been answered. It is three unicode
    normalisations and a per-character category scan deep, which is why it was a third of the cost
    of indexing a corpus with no Sanskrit in it at all. Pure function of `s`, so the cache is only
    ever a memo — same output, 5.5x fewer of them computed.'''
    s = strip_vedic(s)
    return _latn_fold(deva2ascii(s) if DEVANAGARI.search(s) else s)

_VOCALIC_R = re.compile(r'ri(?=[bcdfghjklmnpqstvxz])')   # Krishna's `ri` is IAST `ṛ`
_ASPIRATE = re.compile(r'(?<=[bcdfgjklmnpqrstvxz])h')    # sh, th, dh, bh, ch: the h goes
_LONG, _DOUBLE = re.compile(r'([aeiou])\1+'), re.compile(r'(.)\1+')
_ENDINGS = ('ies', 'ied', 'ing', 'ed', 'es', 's', 'y')

@lru_cache(maxsize=1<<16)
def roman_fold(s:str,
               doubles:bool=False,  # a doubled letter single: Lakshmana and Lakṣmaṇa, Raghuvamsha and Raghuvaṃśa
               endings:bool=False   # an English plural or past ending off, leaving three letters: duties and duty
              ) -> str:
    "`fold_token` plus plain-English spellings: sh/ṣ/ś to s, th/ṭh to t, ee/ī to i, oo/ū to u, aa/ā to a, w to v."
    k = _VOCALIC_R.sub('r', fold_token(s).replace('w', 'v'))
    k = _LONG.sub(r'\1', _ASPIRATE.sub('', k).replace('ee', 'i').replace('oo', 'u'))
    if doubles: k = _DOUBLE.sub(r'\1', k)
    if endings: k = next((k[:-len(e)] for e in _ENDINGS if k.endswith(e) and len(k) - len(e) >= 3), k)
    return k


In [ ]:
test_eq(strip_vedic('अ॒ग्निमी॑ळे'), 'अग्निमीळे')
test_eq(fold_token('भीष्म'), 'bhisma')
test_eq(fold_token('Bhīṣma'), 'bhisma')
test_eq(roman_fold('Bheeshma'), 'bisma')
test_eq(detect_script('धर्म'), 'deva')

In [ ]:
#| export
CITE_RE = re.compile(r'(?:\|\||//)\s*([A-Za-zĀ-ſ]+)[_\s.]([\d,.\-]+[a-z]?(?:\[[^\]\n]{0,16}\])?)\s*(?:\|\||//)')
def cite_parts(c:str) -> tuple:
    "`'BrhUp_1,1.2'` -> `('BrhUp', ['1','1','2'])` — the siglum and its hierarchy."
    c = re.sub(r'\[[^\]]*\]', '', (c or '')).strip()   # drop a `[57M]` alternate-edition number
    m = re.match(r'([^\s_]+)[_\s]([\d,.\-]+[a-z]?)$', c) or re.match(r'([A-Za-zĀ-ſ]+)[.]([\d,.\-]+[a-z]?)$', c)
    if not m: return (c, [])
    return m.group(1), [p for p in re.split(r'[,.]', m.group(2)) if p]

In [ ]:
test_eq(cite_parts('BrhUp_1,1.2'), ('BrhUp', ['1', '1', '2']))
assert CITE_RE.search('atha || MBh_1,1.1 ||')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()